# 3 · HyperSpot pipeline: embeddings → ct-domains → ct-SVGs

The pipeline runs from the **`hyperspot` CLI**. Run the cells below from `tutorial/` (the CLI looks for
`./hyperspot_project_files/<project>`), or paste the commands into a terminal in the same folder.

| step | command | output (in the project folder) |
|---|---|---|
| 1 | `train-embedder` | trained hypergraph autoencoder |
| 2 | `make-embedding` | per-cell (node) and per-spot (hyperedge) embeddings |
| 3 | `compute-spatial-domains` | tissue domains: `ST_aggregated.h5ad` |
| 4 | `compute-celltype-spatial-domains` | **ct-domains**: `obs['domain']` in each `ct_spatial_data/ST_cell_<ct>.h5ad` |
| 5 | `compute-svgs` | tissue SVGs: `svgs/svgs_ST.csv` |
| 6 | `compute-ct-svgs` | **ct-SVGs** (detection): `svgs/svgs_ct_<ct>.csv` |
| 7 | `compute-ct-svgs --ctsvg.gene-scores` (optional) | per-gene table: `svgs/svgs_ct_<ct>_scores.csv` |

Selection thresholds (paper defaults): a gene is a ct-SVG of a domain if Wilcoxon BH-adjusted p < 0.05,
log2FC ≥ 1.5, and it is expressed in ≥ 80 % of the domain's cells.

Kernel: `HyperSpot`. A GPU is needed for steps 1–2 (≈ 1000 epochs).

**GPU selection.** Multi-GPU training is not yet supported, so HyperSpot must see exactly one GPU.
On a machine with one GPU, `--accelerator cuda` uses it and nothing needs to be set. On a machine with
several GPUs, set `CUDA_VISIBLE_DEVICES` to a single GPU number (e.g. `0` or `1`, not `0,1`) before
running; otherwise the commands stop with "Multi-GPU support not implemented". The setup cell below
does this (GPU `0`) unless you already set it.

**Reproducibility.** Seeds are set once at the start of a program, not inside pipeline functions.
Each CLI command is its own program and seeds itself (default 42), so steps 1–2 are reproducible as run
here. `make-embedding` samples neighbourhoods (70 % of hyperedges/cells per hop), so its output depends
on that seed. If you call the steps from Python instead (notebook / API), seed once before the first step:
`from hyperspot.utils import set_seed; set_seed(42)`, then run the steps in the same order.
`perform-model-evaluation` reseeds every run itself (fixed seeds 42, 271, …).

In [ ]:
import os
import pandas as pd

from pathlib import Path

# All tutorial outputs live next to these notebooks; run the notebooks from tutorial/.
DATA_DIR      = Path("../demo/data_repo/Human Kidney Cancer")     # raw inputs (see README)
PATIENT_ID    = "PD47171"
PROJECT       = "Human Kidney Cancer Core"
SPOTIPHY_DIR  = Path("spotiphy_project_files") / PROJECT          # Spotiphy inputs/outputs
HYPERSPOT_DIR = Path("hyperspot_project_files") / PROJECT         # HyperSpot project (the CLI resolves
                                                                  # --project relative to ./hyperspot_project_files)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # one GPU only; multi-GPU is not yet supported
assert (HYPERSPOT_DIR / "hypergraph.pth").exists(), "run notebook 02 first"

## 1–2 · Train the embedder and make embeddings

In [ ]:
!hyperspot train-embedder --project "{PROJECT}" --model.model-name CTHGCAEv6Embedder --model.embedding-dim 20 --model.dropout 0.2 --num-epochs 1000 --batch-size 8192 --accelerator cuda

In [ ]:
!hyperspot make-embedding --project "{PROJECT}" --model.model-name CTHGCAEv6Embedder --model.embedding-dim 20 --model.dropout 0.2 --batch-size 8192 --accelerator cuda

## 3–4 · Tissue domains and cell-type-specific domains (Leiden on the embeddings)

In [ ]:
!hyperspot compute-spatial-domains --project "{PROJECT}"
!hyperspot compute-celltype-spatial-domains --project "{PROJECT}"

## 5–6 · Detect SVGs and ct-SVGs

In [ ]:
!hyperspot compute-svgs --project "{PROJECT}"
!hyperspot compute-ct-svgs --project "{PROJECT}"

## 7 · Per-gene table (optional, for benchmarking)

Step 6 writes only the ct-SVGs. With `--ctsvg.gene-scores`, `compute-ct-svgs` also writes, per cell type,
a table of **every** gene: its best domain, BH-adjusted p-value (`adj_pval`), raw p-value
(`CombinedPvals`) and whether it passes the expression filter (`passes_filter`: ≥ 80 % of the domain's
cells and log2FC ≥ 1.5). Genes that fail the filter get `adj_pval` = 1 and no raw p-value. This is the
format used to benchmark against other ct-SVG methods. A gene with `adj_pval` < 0.05 is exactly a ct-SVG
from step 6.

The command repeats step 6 (same ct-SVG lists) and adds the `*_scores.csv` tables.

In [ ]:
!hyperspot compute-ct-svgs --project "{PROJECT}" --ctsvg.gene-scores

In [ ]:
scores = pd.read_csv(HYPERSPOT_DIR / "svgs" / "svgs_ct_RCC_scores.csv", index_col=0)
print(f"{len(scores)} genes, {scores['passes_filter'].sum()} pass the filter, "
      f"{(scores['adj_pval'] < 0.05).sum()} RCC ct-SVGs")
scores.sort_values("adj_pval").head()

## Optional · Reproducibility runs

`perform-model-evaluation` repeats training → domains → ct-SVGs with 10 fixed seeds into
`evaluation/run_<i>/`. It can resume, and is used by notebook 04 for run-to-run stability.
This takes about 10× steps 1–6.

In [ ]:
# !hyperspot perform-model-evaluation --project "{PROJECT}" --model.model-name CTHGCAEv6Embedder --model.embedding-dim 20 --model.dropout 0.2 --num-epochs 1000 --batch-size 8192 --accelerator cuda --num-runs 10